# Anatomy of a prompt: roles, instruction, context, format

**Session 1 · Foundations · small model (`llama3.2:3b`)**

Turn a vague prompt into a specific one and *measure* the difference: a specific prompt
is one whose output you can check against the constraints you asked for.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from utils import ask, chat, SMALL_MODEL


### Vague vs specific

Same source text, two prompts. The vague one leaves every choice to the model; the
specific one names the role, the audience, the length, and the shape of the output.

In [ ]:
TEXT = (
    "Our Q3 support load rose 22% after the mobile launch. Most tickets were about "
    "login failures on Android 13. The on-call rota was thin during the Berlin office "
    "holiday week, so median first response slipped from 4 hours to 11 hours. We have "
    "since added two contractors and a status-page banner for known issues."
)

vague = "Summarize this."
print(ask(f"{vague}\n\n{TEXT}", model=SMALL_MODEL))

In [ ]:
specific = chat([
    {"role": "system", "content": "You are an editor. Be concise."},
    {"role": "user", "content": (
        "Summarize the TEXT in exactly 3 bullet points, each starting with '- ', "
        "60 words total maximum, for a non-technical manager. No jargon.\n\n"
        f'TEXT:\n"""{TEXT}"""'
    )},
], model=SMALL_MODEL)
print(specific)

### Score both against the constraints

"Better" is not a feeling. Define what you asked for as checks, then count how many each
output passes. The vague prompt never agreed to any of these, so it mostly fails them —
that is the point.

In [ ]:
def checks(out):
    lines = [ln for ln in out.strip().splitlines() if ln.strip()]
    bullets = [ln for ln in lines if ln.lstrip().startswith(("-", "*", "•"))]
    return {
        "3 bullets":       len(bullets) == 3,
        "<= 60 words":     len(out.split()) <= 60,
        "no follow-up Q":  "?" not in out,
        "no code fence":   "```" not in out,
    }

vague_out = ask(f"Summarize this.\n\n{TEXT}", model=SMALL_MODEL)
for name, out in [("vague", vague_out), ("specific", specific)]:
    c = checks(out)
    failed = [k for k, ok in c.items() if not ok]
    print(f"{name:9} {sum(c.values())}/4 checks pass" +
          (f"   (fails: {', '.join(failed)})" if failed else ""))

### Worked example

A prompt built deliberately from the four parts — role, instruction, context, format — and
scored with the same `checks()`. Change one part at a time and watch the output move.

In [4]:
# Worked example: a prompt assembled from role / instruction / context / format
TEXT = """The Apollo program ran from 1961 to 1972. It landed twelve astronauts
on the Moon across six successful missions. At its peak it consumed about 4% of
the US federal budget."""

msgs = [
    # ROLE
    {"role": "system", "content": "You are a briefing assistant for busy executives."},
    # INSTRUCTION + CONTEXT + FORMAT
    {"role": "user", "content": (
        "INSTRUCTION: summarise the TEXT for a non-technical manager.\n"
        "FORMAT: exactly 3 bullets, each starting with '- ', max 15 words each, no jargon.\n\n"
        f'CONTEXT / TEXT:\n"""{TEXT}"""'
    )},
]
print(chat(msgs))


- Ran from 1961 to 1972, achieving Moon landings.  
- Six missions placed twelve astronauts on the Moon.  
- At its height, cost about 4% of the federal budget.


## Your turn - vary the example

1. Label each line of the worked prompt: role / instruction / context / format.
2. Change only the FORMAT line (e.g. "one sentence" or "a JSON list") and re-run.
3. Change only the ROLE and see how tone shifts with everything else fixed.
